# Pooled TOST across seeds (eval-only, ~5–8 min on T4)
Per-seed Freiburg TOST was underpowered (n=1,018). This pools the three seeds correctly: for every test image, correctness is averaged across seeds (ARC-D at each seed's val-selected τ vs always-heavy), then a paired bootstrap runs on the per-image seed-averaged differences. Reports pooled TOST(±1%) for **all four conditions**, both datasets if checkpoints exist.

Needs: GPU runtime, Drive containing `arcd_final/final_ckpts/` and the results CSV(s). **No training.**

In [ ]:
DATASETS = ["freiburg"]        # add "gsd" for the symmetric pooled numbers if its ckpts are on Drive
!pip -q install timm
import os, random, csv, glob
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import cv2, timm
device="cuda" if torch.cuda.is_available() else "cpu"
assert device=="cuda", "use a GPU runtime (eval only, but CPU is too slow)"
from google.colab import drive; drive.mount("/content/drive")
BASE="/content/drive/MyDrive/arcd_final"; CKPT=os.path.join(BASE,"final_ckpts")
print(sorted(os.listdir(CKPT)))

In [ ]:
import urllib.request, subprocess
def load_data(ds):
    if ds=="gsd":
        if not os.path.exists("GroceryStoreDataset"):
            subprocess.run(["git","clone","-q","https://github.com/marcusklasson/GroceryStoreDataset.git"],check=True)
        R="GroceryStoreDataset/dataset"
        def rd(t):
            out=[]
            with open(os.path.join(R,t)) as f:
                for line in f:
                    line=line.strip()
                    if not line: continue
                    p=[x.strip() for x in line.split(",")]
                    out.append((os.path.join(R,p[0]),int(p[1])))
            return out
        return rd("test.txt"), 81
    if not os.path.exists("images"):
        url="http://aisdatasets.informatik.uni-freiburg.de/freiburg_groceries_dataset/freiburg_groceries_dataset.tar.gz"
        print("downloading (~1GB)..."); urllib.request.urlretrieve(url,"fg.tar.gz")
        subprocess.run(["tar","-xf","fg.tar.gz"],check=True)
    RAW="https://raw.githubusercontent.com/PhilJd/freiburg_groceries_dataset/master/splits"
    if not os.path.exists("test0.txt"): urllib.request.urlretrieve(f"{RAW}/test0.txt","test0.txt")
    seen=set(); out=[]
    with open("test0.txt") as fh:
        for line in fh:
            line=line.strip()
            if not line: continue
            rel,lab=line.rsplit(" ",1)
            if rel in seen: continue
            seen.add(rel); out.append((os.path.join("images",rel),int(lab)))
    return out, 25

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
def to_t(img):
    t=torch.from_numpy(img).float().permute(2,0,1)/255.0
    for c in range(3): t[c]=(t[c]-MEAN[c])/STD[c]
    return t
def corrupt(img,kind,sev,idx=0):
    if kind=="none" or sev==0: return img
    if kind=="blur":
        k=[3,5,7,11,15][sev-1]; return cv2.GaussianBlur(img,(k,k),0)
    if kind=="jpeg":
        q=[40,30,20,12,7][sev-1]
        _,e=cv2.imencode(".jpg",cv2.cvtColor(img,cv2.COLOR_RGB2BGR),[cv2.IMWRITE_JPEG_QUALITY,q])
        return cv2.cvtColor(cv2.imdecode(e,cv2.IMREAD_COLOR),cv2.COLOR_BGR2RGB)
    if kind=="occlusion":
        s=[0.1,0.2,0.3,0.4,0.5][sev-1]; side=int(224*s)
        r=random.Random(10_000*sev+idx)
        x0=r.randint(0,224-side); y0=r.randint(0,224-side)
        img=img.copy(); img[y0:y0+side,x0:x0+side]=127; return img
class TDS(Dataset):
    def __init__(self,items,kind,sev): self.items,self.kind,self.sev=items,kind,sev
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        p,l=self.items[i]
        img=np.array(Image.open(p).convert("RGB").resize((224,224)))
        return to_t(corrupt(img,self.kind,self.sev,idx=i)), l

In [ ]:
# taus per (dataset,seed) from the results CSVs on Drive (ARC-D rows)
def load_taus():
    taus={}
    for f in glob.glob(os.path.join(BASE,"*.csv")):
        with open(f) as fh:
            for r in csv.DictReader(fh):
                if r["system"]=="ARC-D" and r["condition"]=="clean" and r.get("tau"):
                    taus[(r["dataset"],int(r["seed"]))]=float(r["tau"])
    return taus
TAUS=load_taus(); print("taus:",TAUS)

@torch.no_grad()
def eval_model(model,items,kind,sev,bs=64):
    model.eval()
    dl=DataLoader(TDS(items,kind,sev),batch_size=bs,num_workers=2,pin_memory=True)
    C,K=[],[]
    for x,y in dl:
        p=torch.softmax(model(x.to(device)),1); c,pred=p.max(1)
        C.append(c.cpu().numpy()); K.append((pred.cpu()==y).numpy())
    return np.concatenate(C),np.concatenate(K)

def boot(diff,n=5000):
    idx=np.random.randint(0,len(diff),size=(n,len(diff)))
    b=diff[idx].mean(axis=1); return diff.mean(),np.percentile(b,5),np.percentile(b,95)

CONDS=[("clean","none",0),("jpeg_s2","jpeg",2),("blur_s2","blur",2),("occ_s1","occlusion",1)]
for ds in DATASETS:
    items,NC=load_data(ds)
    print(f"\n===== {ds} (n={len(items)}) pooled over seeds =====")
    per_cond={t:{ "arcd":[], "heavy":[] } for t,_,_ in CONDS}
    for seed in (0,1,2):
        d=timm.create_model("efficientnet_b0",pretrained=False,num_classes=NC).to(device)
        d.load_state_dict(torch.load(os.path.join(CKPT,f"{ds}_distilled_seed{seed}.pt"),map_location=device))
        h=timm.create_model("convnext_tiny",pretrained=False,num_classes=NC).to(device)
        h.load_state_dict(torch.load(os.path.join(CKPT,f"{ds}_heavy_seed{seed}.pt"),map_location=device))
        tau=TAUS[(ds,seed)]
        for tag,kind,sev in CONDS:
            cd,kd=eval_model(d,items,kind,sev)
            _,kh=eval_model(h,items,kind,sev)
            karcd=np.where(cd>=tau,kd,kh)
            per_cond[tag]["arcd"].append(karcd.astype(float))
            per_cond[tag]["heavy"].append(kh.astype(float))
        del d,h; torch.cuda.empty_cache()
        print(f"  seed {seed} done (tau={tau:.3f})")
    for tag,_,_ in CONDS:
        a=np.mean(per_cond[tag]["arcd"],axis=0)   # per-image mean over seeds
        hh=np.mean(per_cond[tag]["heavy"],axis=0)
        m,lo,hi=boot(a-hh)
        v="EQUIVALENT" if (lo>-0.01 and hi<0.01) else "not equivalent"
        print(f"  {tag:8s} pooled ARC-D - heavy: {m*100:+.2f}%  90%CI [{lo*100:+.2f},{hi*100:+.2f}]  -> {v}")

Paste the pooled lines back. Paper wording if clean pools EQUIVALENT: *"Per-seed equivalence tests on Freiburg were underpowered (n=1,018); pooling per-image outcomes across the three seeds yields equivalence within ±1% (TOST, 90% CI)."* If it stays outside, we report exactly that — point estimates consistent, equivalence not established at this n.